# 📘 Tox21 — Notebook Penjelasan Lengkap

> **Notebook ini adalah versi beranotasi dari [`Tox21.ipynb`](./Tox21.ipynb).**
> Setiap cell kode didahului oleh penjelasan markdown:
> - **Apa yang dilakukan cell** — penjelasan logika kode dalam bahasa awam
> - **Kenapa dibutuhkan** — peran cell dalam pipeline keseluruhan
> - **Cara baca outputnya** — terutama untuk cell visualisasi (kurva ROC, heatmap, dll)
>
> Tujuannya supaya orang yang baru belajar machine learning untuk toksikologi
> bisa memahami **apa, kenapa, dan bagaimana** dari tiap langkah, bukan hanya
> menjalankan kode tanpa konteks.

---

## 🗺️ Peta Notebook

| Bagian | Cell | Topik |
|--------|------|-------|
| **A.** Pengenalan | 0–1 | Konsep prediksi toksisitas |
| **B.** Setup | 2–3 | Instalasi & impor library |
| **C.** Data | 4–5 | Load Tox21 + balancing |
| **D.** Helper | 6–7 | Fungsi evaluasi + training |
| **E.** Training | 8–12 | GCN & GAT model |
| **F.** Evaluasi | 13–14 | Hitung metrik per model |
| **G.** Visualisasi | 15–22 | 6 plot interpretasi hasil |
| **H.** Analisis | 23–24 | Per-task breakdown |
| **I.** Export | 25–26 | Artifacts untuk API backend |
| **J.** Diskusi | 27–29 | Tanya-jawab konseptual |
| **K.** Demo | 30 | Prediksi pada SMILES baru |

---

## 🎯 Glosarium Singkat

- **SMILES** — notasi teks untuk struktur molekul (mis. aspirin = `CC(=O)Oc1ccccc1C(=O)O`)
- **Endpoint toksikologi** — jalur biologis yang diuji (12 endpoint Tox21: 7 reseptor nuklir + 5 stress response)
- **Multi-task classification** — satu model memprediksi 12 label biner sekaligus
- **GNN** — Graph Neural Network; memperlakukan molekul sebagai graf (atom=node, bond=edge)
- **GCN** — Graph Convolutional Network; agregasi rata-rata dari tetangga
- **GAT** — Graph Attention Network; agregasi dengan bobot perhatian yang dipelajari
- **ROC-AUC** — area di bawah kurva ROC; ukuran kemampuan ranking model (0.5 = random, 1.0 = sempurna)
- **AUPRC** — area di bawah kurva Precision-Recall; lebih informatif untuk dataset imbalanced


# Prediksi Toksisitas Molekul dengan GNN pada Dataset Tox21

Notebook ini melatih dua model Graph Neural Network (GCNModel dan GATModel) menggunakan DeepChem 2.8
untuk memprediksi toksisitas molekul dari dataset Tox21 (12 endpoint toksikologi).

**Environment:** DeepChem 2.8 | PyTorch 2.5.1+cu121 | Python 3.10 | DGL 2.4.0+cu121  
**Model:** GCNModel, GATModel — berjalan di GPU (NVIDIA GTX 1650) jika CUDA tersedia, fallback ke CPU

## A. Pemahaman Konseptual: Prediksi Toksisitas Molekuler

### A.1 Mengapa Prediksi Toksisitas Sulit Tanpa Komputasi?

Tantangan utama bukan kompleksitas kimia semata — melainkan kombinasi antara **ruang pencarian yang tidak terbatas** dan **biaya eksperimental yang tidak skalabel**. Ruang kimia sintetis yang dapat dijelajahi diperkirakan mencakup sekitar $10^{60}$ molekul potensial, sementara seluruh senyawa yang pernah disintesis manusia hanya mencapai sekitar $10^8$ — kurang dari satu-triliun-triliun bagian dari ruang yang ada. Mencari senyawa toksik secara eksperimental dalam ruang sebesar itu adalah masalah pencarian yang secara fundamental tidak dapat diselesaikan tanpa panduan komputasional.

Dari sisi praktis, biaya pengujian toksisitas in vitro berkisar antara \$1.000–\$10.000 per senyawa *per assay*. Tox21 menguji **12 endpoint secara bersamaan** — berarti biaya per molekul bisa mencapai puluhan ribu dolar. Untuk 7.804 molekul dalam dataset ini saja, pengujian eksperimental lengkap membutuhkan biaya di kisaran ratusan juta dolar, menjadikannya mustahil sebagai pendekatan awal penapisan senyawa.

Yang lebih fundamental secara statistik: distribusi label positif sangat tidak seimbang. Rata-rata hanya **8–12% sampel bersifat toksik** untuk setiap endpoint, dengan beberapa task seperti NR-PPAR-γ memiliki *positive rate* di bawah 3%. Model naif yang selalu memprediksi "non-toksik" sudah mencapai akurasi >90% tanpa mempelajari apapun yang berguna. Itulah alasan `BalancingTransformer` dibutuhkan dalam pipeline ini — untuk mengkalibrasi ulang bobot sampel sehingga minoritas positif tidak tenggelam dalam sinyal mayoritas negatif. Tanpa koreksi ini, optimisasi loss akan memilih jalan pintas prediksi konstan.

Yang paling membuat prediksi sulit secara mekanistik adalah sifat **emergent** dari toksisitas: tidak ada satu substruktur pun yang secara deterministik memprediksi aktivitas toksikologi. Benzena bersifat karsinogenik melalui epoksidasi oleh CYP1A2, tetapi turunan benzena tersubstitusi tertentu aman dalam dosis terapeutik. Toksisitas muncul dari **interaksi kolektif** antara gugus fungsional, topologi molekuler global, dan komplementaritas geometrik dengan situs aktif protein — sebuah properti yang tidak dapat direduksi menjadi pencarian substruktur sederhana.

---

### A.2 Mengapa Pendekatan Komputasional Berbasis Graf?

Representasi graf bukan analogi untuk molekul — ini adalah representasi **identitas**. Ikatan kimia secara definisi adalah relasi biner antara pasangan atom, yang merupakan definisi tepat dari *edge* dalam graf berbobot. Atom adalah node dengan atribut (nomor atom, hibridisasi, muatan). Ketika kita menulis molekul sebagai $G = (V, E)$, kita tidak menyederhanakan struktur — kita merepresentasikannya secara lossless pada level topologi 2D.

Bandingkan dengan fingerprint berbasis hashing seperti Morgan atau MACCS. Fingerprint mengkodekan **kehadiran substruktur** sebagai bit tunggal, tetapi informasi posisi hilang sepenuhnya: gugus karbonil $\mathrm{C{=}O}$ di ujung rantai alifatik dan $\mathrm{C{=}O}$ di tengah cincin aromatik terkonjugasi mengaktifkan bit yang sama, padahal reaktivitas elektrofilnya berbeda drastis. Dua molekul dengan fingerprint Morgan sangat mirip bisa memiliki toksisitas sangat berbeda jika substruktur identik tersebut dikelilingi oleh konteks elektronik yang berbeda — konteks yang tidak pernah masuk ke vektor input fingerprint.

GNN mempertahankan **positional context** selama agregasi: representasi setiap atom pada layer $l$ adalah fungsi eksplisit dari embedding tetangganya, sehingga gugus karbonil dalam dua konteks berbeda menghasilkan embedding berbeda setelah satu layer agregasi. Kemampuan ini secara langsung terukur dalam eksperimen ini: GCN mencapai Test ROC-AUC **0.8148** sementara MLP terbaik hanya mencapai **0.7594** — selisih 0.055 yang sepenuhnya dapat diatribusikan pada kemampuan GCN mempertahankan konteks lokal yang hilang dalam representasi fingerprint flat.

---

### A.3 Relevansi Biologis 12 Endpoint Tox21

Dua belas endpoint Tox21 dipilih karena mewakili dua kategori mekanisme toksikologi yang paling relevan secara regulatori.

**Kelompok Nuclear Receptor (NR-)** mencakup reseptor inti yang, ketika teraktivasi atau terinhibisi secara tidak tepat oleh senyawa asing, mendisrupsi sinyal hormonal dan metabolisme:
- **NR-AR / NR-AR-LBD**: Androgen receptor dan *ligand-binding domain*-nya — aktivasi menyebabkan disrupsi endokrin pada sistem reproduktif.
- **NR-ER / NR-ER-LBD**: Estrogen receptor — relevan untuk deteksi *endocrine disruptors* yang terkait risiko kanker payudara.
- **NR-AhR**: Aryl hydrocarbon receptor — menginduksi ekspresi CYP1A1/1A2 yang mengaktifkan prokarsinogen polisiklik.
- **NR-Aromatase**: Enzim CYP19A1 yang mengkonversi androgen ke estrogen — inhibisi mendisrupsi rasio hormon seks.
- **NR-PPAR-γ**: Mengatur diferensiasi adiposit dan metabolisme lipid.

**Kelompok Stress Response (SR-)** mengukur aktivasi jalur seluler yang dipicu oleh kerusakan atau stres:
- **SR-ARE** via Nrf2: marker stres oksidatif dan deplesi glutathion.
- **SR-ATAD5**: Rekrutmen protein perbaikan DNA akibat kerusakan heliks.
- **SR-HSE**: Aktivasi Heat Shock Proteins akibat misfolding protein.
- **SR-MMP**: Gangguan Mitochondrial Membrane Potential — prekursor apoptosis.
- **SR-p53**: Aktivasi tumor suppressor p53 oleh kerusakan DNA berat.

Relevansi langsung untuk pemodelan multi-task adalah korelasi biologis antar endpoint: NR-AR dan NR-AR-LBD berbagi pocket pengikatan yang sama — satu menggunakan reseptor penuh, satu domain terisolasi. Gradient update dari satu task memberikan informasi yang *genuine* untuk task lain yang berbagi mekanisme, bukan sekadar regularisasi. Ini menjustifikasi secara biologis mengapa multi-task learning lebih dari sekadar teknik regularisasi dalam konteks ini.

### 🧰 Cell 2 — Instalasi Dependensi

**Apa yang dilakukan:** Meng-install paket Python yang diperlukan untuk pipeline ini lewat `pip`:
- **DeepChem** — framework ML untuk drug discovery (menyediakan dataset Tox21, model GCN/GAT, metric)
- **DGL** (Deep Graph Library) — backend graph computation untuk GNN
- **RDKit** — toolkit kimia komputasi (parsing SMILES, fingerprint, struktur 2D)
- **scikit-learn, matplotlib, seaborn, pandas** — utilitas standar Python data science

**Kenapa dibutuhkan:** DeepChem mengelola seluruh pipeline (load data → featurize → train → evaluate). DGL adalah engine yang menjalankan operasi GNN-nya di GPU. RDKit dibutuhkan untuk validasi SMILES dan visualisasi struktur.

> 💡 **Catatan:** Kalau sudah pernah install dependency-nya, cell ini akan cepat (cek paket sudah ada lalu skip).


In [ ]:
# Instalasi dependensi yang diperlukan
# DGL CUDA 12.1 dibutuhkan oleh GCNModel dan GATModel agar bisa jalan di GPU
%pip install -q deepchem rdkit torch_geometric 2>/dev/null || true

# Install DGL versi CUDA 12.1 (sesuai PyTorch 2.5.1+cu121)
# Wheel diambil dari folder torch-2.3 karena folder torch-2.5 masih restricted di CDN DGL
%pip install -q "https://data.dgl.ai/wheels/torch-2.3/cu121/dgl-2.4.0%2Bcu121-cp310-cp310-manylinux1_x86_64.whl" 2>/dev/null || \
%pip install -q dgl  # fallback ke CPU jika download CUDA gagal

### 🧰 Cell 3 — Import Library

**Apa yang dilakukan:** Mengimpor semua modul yang akan dipakai sepanjang notebook:
- `dc` — alias DeepChem
- `torch` — backend PyTorch (DGL pakai PyTorch)
- `np`, `pd` — NumPy & Pandas untuk manipulasi data
- `plt`, `sns` — matplotlib & seaborn untuk plotting
- `roc_curve`, `auc`, dll — metrik scikit-learn untuk evaluasi

**Kenapa `%matplotlib inline`?** Memastikan plot tampil langsung di output cell notebook (bukan window terpisah atau file PNG).

**Variabel `DEVICE`:** Mendeteksi apakah ada GPU NVIDIA (CUDA). Kalau ada → training pakai GPU (jauh lebih cepat); kalau tidak → fallback ke CPU.


In [ ]:
# Mengatur variabel lingkungan sebelum import deepchem untuk kompatibilitas Keras
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Import pustaka utama
import deepchem as dc
import torch
import numpy as np
import pandas as pd

# Plot tampil inline di notebook (tidak disimpan ke file PNG)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Metrik sklearn untuk kurva ROC dan Precision-Recall per task
from sklearn.metrics import roc_curve, precision_recall_curve, auc, roc_auc_score

# Deteksi otomatis perangkat: gunakan GPU jika CUDA tersedia, fallback ke CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"DeepChem versi : {dc.__version__}")
print(f"PyTorch versi  : {torch.__version__}")
print(f"CUDA tersedia  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device digunakan: {DEVICE}")

### 📦 Cell 4 — Load Dataset Tox21

**Apa yang dilakukan:**
1. Mengunduh dataset Tox21 dari MoleculeNet (kalau belum ada di cache lokal)
2. Memfeaturisasi setiap molekul menjadi **graf** (node = atom, edge = bond) dengan `MolGraphConvFeaturizer(use_edges=True)`
3. Membagi dataset random 80% train / 10% valid / 10% test

**Tox21 dataset:**
- ~7,800 molekul kimia
- 12 endpoint biologis (label biner: toksik=1, non-toksik=0)
- Dataset *sparse*: tidak semua molekul punya label untuk semua 12 endpoint

**Output yang dicetak:**
- Daftar 12 nama task (`NR-AR`, `NR-AhR`, ..., `SR-p53`)
- Ukuran masing-masing split

> ℹ️ Splitter random ≠ paling realistis untuk drug discovery (idealnya `scaffold split`), tapi konsisten dengan benchmark Tox21Challenge.


In [ ]:
# Memuat dataset Tox21 dari MoleculeNet menggunakan MolGraphConvFeaturizer
# use_edges=True: menyertakan fitur tepi (bond features) dalam representasi graf
# splitter='random': membagi data secara acak (~80/10/10)
print("Memuat dataset Tox21 dengan MolGraphConvFeaturizer...")

tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer=dc.feat.MolGraphConvFeaturizer(use_edges=True),
    splitter='random'
)

train_dataset, valid_dataset, test_dataset = datasets

print(f"\nDaftar tasks (12 endpoint toksikologi):")
for i, t in enumerate(tasks):
    print(f"  {i+1:2d}. {t}")

print(f"\nUkuran dataset:")
print(f"  Train : {len(train_dataset)} molekul")
print(f"  Valid : {len(valid_dataset)} molekul")
print(f"  Test  : {len(test_dataset)} molekul")

### ⚖️ Cell 5 — Balancing Transformer

**Apa yang dilakukan:** Menerapkan `BalancingTransformer` ke training set.

**Kenapa dibutuhkan:** Tox21 sangat tidak seimbang — biasanya hanya **5–15% molekul positif** (toksik) per task. Kalau dibiarkan, model akan cenderung selalu memprediksi "non-toksik" karena itu sudah benar 85–95% waktu. BalancingTransformer mengubah **bobot sampel** (bukan menduplikasi) sehingga loss function memberi kelas minoritas perhatian setara dengan kelas mayoritas.

**Yang berubah:** hanya `train_dataset.w` (weights). Valid/test set TIDAK ditransformasi — kita mau evaluasi pada distribusi nyata.


In [ ]:
# Menerapkan BalancingTransformer pada data latih untuk menangani ketidakseimbangan kelas
# Transformer ini menyesuaikan bobot sampel agar kelas minoritas (toksik) lebih diperhatikan
print("Menerapkan BalancingTransformer pada data latih...")

transformer = dc.trans.BalancingTransformer(dataset=train_dataset)
train_dataset = transformer.transform(train_dataset)

print("BalancingTransformer berhasil diterapkan.")
print(f"Tipe fitur: {type(train_dataset.X[0]).__name__}")

### 🔧 Cell 6 — Helper: `evaluate_model()`

**Apa yang dilakukan:** Membuat fungsi reusable untuk menghitung dua metrik utama (ROC-AUC + AUPRC) pada train/valid/test split.

**Dua metrik yang dihitung:**
- **ROC-AUC** — seberapa baik model membedakan kelas positif vs negatif secara umum (skala 0.5–1.0; 0.5 = random)
- **AUPRC** — seberapa baik model dalam menemukan kelas positif (lebih informatif untuk data tidak seimbang)

**Kenapa rata-rata di 12 task?** Tox21 adalah multi-task → satu skor agregat memudahkan perbandingan model. Tapi kita juga akan melihat **per-task breakdown** nanti (Cell 18-19).


In [ ]:
# Fungsi pembantu: menghitung ROC-AUC dan AUPRC untuk semua split dataset
def evaluate_model(model, train_ds, valid_ds, test_ds):
    """Mengevaluasi model pada train/valid/test dan mengembalikan dict metrik."""
    # Mendefinisikan metrik rata-rata di semua 12 task
    metric_roc = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean)
    metric_prc = dc.metrics.Metric(dc.metrics.prc_auc_score, np.mean)

    results = {}
    for split_name, ds in [('train', train_ds), ('valid', valid_ds), ('test', test_ds)]:
        roc = model.evaluate(ds, [metric_roc])['mean-roc_auc_score']
        prc = model.evaluate(ds, [metric_prc])['mean-prc_auc_score']
        results[split_name] = {'roc': roc, 'prc': prc}
        print(f"  {split_name:5s} | ROC-AUC: {roc:.4f} | AUPRC: {prc:.4f}")

    return results

### 🔧 Cell 7 — Helper: `train_with_early_stopping()`

**Apa yang dilakukan:** Fungsi untuk training dengan early stopping berbasis valid ROC-AUC.

**Logika:**
1. Train model dalam blok kecil (misal: 10 epoch per blok)
2. Setiap akhir blok → cek ROC-AUC pada valid set
3. Simpan checkpoint jika lebih baik dari sebelumnya
4. Setelah `total_epochs` selesai → restore checkpoint dengan valid ROC tertinggi

**Kenapa early stopping?** Mencegah overfitting — model lanjut training meski sudah mulai memorize training set. Checkpoint terbaik = titik di mana model paling generalize ke data baru.

**Output `history`:** list dict `[{epoch, valid_roc}, ...]` yang dipakai untuk plot kurva pelatihan nanti.


In [ ]:
# Fungsi pembantu: pelatihan dengan early stopping berbasis valid ROC-AUC
def train_with_early_stopping(model, train_ds, valid_ds,
                               total_epochs=60, check_every=10, model_name="model"):
    """
    Melatih model dalam blok check_every epoch dan menyimpan checkpoint terbaik
    berdasarkan nilai valid ROC-AUC tertinggi.
    """
    best_valid_roc = -1.0
    best_epoch = 0
    history = []

    # Metrik untuk monitoring validasi setiap blok epoch
    metric_roc = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean)

    # Direktori checkpoint - path stabil agar API backend bisa load model
    project_root = os.path.dirname(os.path.abspath(os.getcwd() + "/Tox21.ipynb"))
    checkpoint_dir = os.path.join(project_root, "checkpoints", f"{model_name.lower()}_best")
    os.makedirs(checkpoint_dir, exist_ok=True)

    n_blocks = total_epochs // check_every
    print(f"\n{'='*60}")
    print(f" Melatih {model_name} | {total_epochs} epoch total (cek setiap {check_every} epoch)")
    print(f"{'='*60}")

    for block in range(n_blocks):
        current_epoch = (block + 1) * check_every

        # Melanjutkan pelatihan - DeepChem tidak mereset bobot antar pemanggilan fit()
        model.fit(train_ds, nb_epoch=check_every)

        # Evaluasi pada data validasi setelah setiap blok
        valid_roc = model.evaluate(valid_ds, [metric_roc])['mean-roc_auc_score']
        history.append({'epoch': current_epoch, 'valid_roc': valid_roc})

        # Tandai dan simpan jika ini merupakan epoch terbaik
        marker = ''
        if valid_roc > best_valid_roc:
            best_valid_roc = valid_roc
            best_epoch = current_epoch
            model.save_checkpoint(model_dir=checkpoint_dir)
            marker = ' <-- checkpoint terbaik disimpan'

        print(f"  Epoch {current_epoch:3d}/{total_epochs} | Valid ROC-AUC: {valid_roc:.4f}{marker}")

    # Memulihkan bobot model terbaik dari checkpoint yang tersimpan
    try:
        model.restore(model_dir=checkpoint_dir)
        print(f"\n  [OK] Dipulihkan dari epoch terbaik: {best_epoch} (ROC-AUC={best_valid_roc:.4f})")
    except Exception as e:
        print(f"\n  [WARN] Tidak dapat memulihkan checkpoint: {e}")
        print(f"         Menggunakan bobot dari epoch terakhir.")

    return history

## Model 1: Graph Convolutional Network (GCN)

GCNModel menggunakan lapisan graph convolution untuk mengagregasi informasi dari atom tetangga.
Arsitektur: 2 lapisan GCN dengan 128 unit masing-masing, diikuti oleh lapisan klasifikasi.

### 🧠 Cell 9 — Training Model GCN

**Apa yang dilakukan:** Membuat dan melatih **Graph Convolutional Network**.

**Hyperparameter:**
- `graph_conv_layers=[128, 128]` — 2 lapisan GCN, masing-masing 128 hidden units
- `dropout=0.3` — regularisasi (30% neuron di-drop saat training)
- `learning_rate=0.001` — laju Adam optimizer
- `batch_size=64` — jumlah molekul per forward pass

**Cara kerja GCN secara intuitif:**
> Setiap atom mengumpulkan informasi dari **atom tetangga** dan mengupdate fitur dirinya. Setelah beberapa lapisan, setiap atom "tahu" lingkungan kimianya sampai 2-3 bond jauhnya. Output akhir adalah **graph-level embedding** yang diteruskan ke classifier 12-task.

**Output cell:** Logging epoch-by-epoch valid ROC-AUC dari `train_with_early_stopping`.


In [ ]:
# Membuat GCNModel dengan konfigurasi yang ditentukan
# device=DEVICE: otomatis pakai GPU jika CUDA tersedia (DGL 2.4.0+cu121 terinstal)
gcn_model = dc.models.GCNModel(
    n_tasks=len(tasks),            # 12 endpoint toksikologi
    graph_conv_layers=[128, 128],  # dua lapisan GCN dengan 128 unit masing-masing
    dropout=0.3,                   # regularisasi dropout 30%
    learning_rate=0.001,           # laju pembelajaran optimizer Adam
    batch_size=64,                 # ukuran batch per iterasi
    mode='classification',         # mode klasifikasi biner multi-task
    device=DEVICE                  # GPU (cuda) atau CPU sesuai deteksi otomatis
)
print(f"GCNModel siap dilatih pada: {gcn_model.device}")

# Melatih dengan early stopping (60 epoch total, pantau setiap 10 epoch)
gcn_history = train_with_early_stopping(
    gcn_model, train_dataset, valid_dataset,
    total_epochs=60, check_every=10, model_name="GCN"
)

### 🩺 Cell 10 — Diagnostik Environment

**Apa yang dilakukan:** Print info GPU/CUDA/DGL untuk troubleshooting kalau training error.

**Tidak kritis untuk pipeline** — hanya untuk memastikan setup hardware sudah benar.


In [ ]:
# === DIAGNOSTIC: cek environment sebelum training ===
import sys
print("Python   :", sys.executable)
try:
    import dgl
    print("DGL      :", dgl.__version__)
except Exception as e:
    print("DGL ERROR:", e)
    print(">> Jalankan cell install ulang, lalu restart kernel")
print("DEVICE   :", DEVICE)
print("Torch    :", __import__('torch').__version__)

## Model 2: Graph Attention Network (GAT)

GATModel menggunakan mekanisme attention untuk memberi bobot berbeda pada tetangga tiap atom.
Arsitektur: 4 attention head, memungkinkan model fokus pada ikatan kimia yang paling relevan.

### 🧠 Cell 12 — Training Model GAT

**Apa yang dilakukan:** Membuat dan melatih **Graph Attention Network**.

**Perbedaan dengan GCN:**
- `n_attention_heads=4` — 4 "attention head" paralel
- GAT bukan rata-rata sederhana, tapi **bobot perhatian dipelajari** untuk tiap pasangan atom

**Intuisi attention:**
> GCN: "Setiap tetangga sama pentingnya."
> GAT: "Tetangga oksigen kemungkinan lebih informatif daripada hidrogen → beri dia bobot lebih tinggi."
>
> Attention head berbeda bisa belajar **pola yang berbeda** (mis. satu fokus ke gugus aromatik, lain ke ikatan rangkap, dst).

GAT umumnya lebih ekspresif tapi lebih sulit dilatih — di proyek ini GCN ternyata sedikit lebih baik.


In [ ]:
# Membuat GATModel dengan konfigurasi yang ditentukan
# n_attention_heads=4: model menggunakan 4 attention head secara paralel
gat_model = dc.models.GATModel(
    n_tasks=len(tasks),      # 12 endpoint toksikologi
    n_attention_heads=4,     # jumlah attention head
    dropout=0.3,             # regularisasi dropout 30%
    learning_rate=0.001,     # laju pembelajaran optimizer Adam
    batch_size=64,           # ukuran batch per iterasi
    mode='classification',   # mode klasifikasi biner multi-task
    device=DEVICE            # GPU (cuda) atau CPU sesuai deteksi otomatis
)
print(f"GATModel siap dilatih pada: {gat_model.device}")

# Melatih dengan early stopping (60 epoch total, pantau setiap 10 epoch)
gat_history = train_with_early_stopping(
    gat_model, train_dataset, valid_dataset,
    total_epochs=60, check_every=10, model_name="GAT"
)

## Evaluasi Model

Membandingkan performa GCN dan GAT dengan baseline MLP pada metrik ROC-AUC dan AUPRC.

### 📊 Cell 14 — Evaluasi Final & Tabel Perbandingan

**Apa yang dilakukan:**
1. Hitung metrik final GCN + GAT pada semua split (train/valid/test)
2. Susun tabel `results_summary` yang membandingkan keduanya dengan baseline MLP (dari eksperimen sebelumnya yang sudah didokumentasikan)

**Tabel `results_summary`:** kolom Train ROC, Valid ROC, Test ROC, Test PRC untuk masing-masing model. Tabel ini yang dipakai untuk bar chart di Cell 16.

> 🔍 **Cara baca tabel:**
> - Train ROC tinggi tapi Test ROC rendah = **overfitting**
> - Train ≈ Valid ≈ Test = **healthy** (generalisasi bagus)
> - Test ROC > 0.75 = umumnya dianggap layak untuk eksplorasi kandidat senyawa


In [ ]:
# Mengevaluasi GCNModel pada semua split (train, valid, test)
print("Evaluasi GCNModel:")
gcn_scores = evaluate_model(gcn_model, train_dataset, valid_dataset, test_dataset)

# Mengevaluasi GATModel pada semua split
print("\nEvaluasi GATModel:")
gat_scores = evaluate_model(gat_model, train_dataset, valid_dataset, test_dataset)

# Menyusun tabel perbandingan lengkap termasuk baseline MLP dari eksperimen sebelumnya
results_summary = pd.DataFrame([
    {'Model': 'MLP v1', 'Train ROC': 0.9937, 'Valid ROC': 0.7732, 'Test ROC': 0.7594, 'Test PRC': 0.3228},
    {'Model': 'MLP v3', 'Train ROC': 0.9932, 'Valid ROC': 0.7721, 'Test ROC': 0.7423, 'Test PRC': 0.3057},
    {
        'Model': 'GCN',
        'Train ROC': gcn_scores['train']['roc'], 'Valid ROC': gcn_scores['valid']['roc'],
        'Test ROC':  gcn_scores['test']['roc'],  'Test PRC': gcn_scores['test']['prc']
    },
    {
        'Model': 'GAT',
        'Train ROC': gat_scores['train']['roc'], 'Valid ROC': gat_scores['valid']['roc'],
        'Test ROC':  gat_scores['test']['roc'],  'Test PRC': gat_scores['test']['prc']
    },
])

# Menampilkan tabel perbandingan dengan format angka yang bersih
pd.set_option('display.float_format', '{:.4f}'.format)
print("\n" + "="*65)
print(" TABEL PERBANDINGAN MODEL")
print("="*65)
print(results_summary.to_string(index=False))

# Menyimpan hasil ke file CSV untuk referensi selanjutnya
results_summary.to_csv('results_summary.csv', index=False)
print("\nHasil perbandingan disimpan ke: results_summary.csv")

## Visualisasi Hasil

Enam jenis visualisasi untuk analisis komprehensif performa model.

### 📈 Cell 16 — Visualisasi 1: Bar Chart Perbandingan Model

**Apa yang ditampilkan:** Dua bar chart **side-by-side**:
- **Kiri:** Test ROC-AUC per model (skala 0.5–1.0)
- **Kanan:** Test AUPRC per model (skala 0.0–0.8)

#### 📖 Cara baca grafik

**Sumbu X (horizontal):** nama model — MLP v1, MLP v3, GCN, GAT

**Sumbu Y (vertikal):**
- Panel kiri: ROC-AUC (semakin tinggi semakin baik; 0.5 = random)
- Panel kanan: AUPRC (semakin tinggi semakin baik; baseline = prevalensi positif data)

**Garis putus-putus abu-abu (kiri):** Random baseline = 0.5 — model di bawah garis ini **lebih buruk dari menebak**.

**Label angka di atas tiap bar:** nilai persis (4 desimal). Ini bobot perbandingan utama.

#### 🎯 Apa yang kita cari?
1. **Bar tertinggi** = model terbaik
2. **Selisih ROC vs AUPRC**: ROC bisa tinggi sementara AUPRC rendah → tanda data tidak seimbang dan model masih kesulitan menangkap kelas minoritas
3. **GNN vs MLP**: idealnya GCN/GAT lebih tinggi → membuktikan struktur molekul sebagai graf > vektor sidik jari biasa

> 💡 Pada dataset ini: **GCN ≈ 0.81** > **GAT ≈ 0.77** > **MLP baseline ≈ 0.76** — GNN unggul tipis tapi konsisten.


In [ ]:
# Visualisasi 1 & 2: Bar chart perbandingan Test ROC-AUC dan Test AUPRC semua model
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Perbandingan Performa Model pada Dataset Tox21 (Test Set)', fontsize=14, fontweight='bold')

# Warna berbeda untuk setiap model agar mudah dibedakan
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
models_list = results_summary['Model'].tolist()

# Panel kiri: Test ROC-AUC
bars1 = axes[0].bar(models_list, results_summary['Test ROC'], color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Test ROC-AUC (rata-rata 12 task)', fontsize=12)
axes[0].set_ylabel('ROC-AUC Score')
axes[0].set_xlabel('Model')
axes[0].set_ylim(0.5, 1.0)
axes[0].axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, label='Random baseline')
axes[0].legend()
# Menambahkan label nilai di atas setiap bar
for bar, val in zip(bars1, results_summary['Test ROC']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# Panel kanan: Test AUPRC
bars2 = axes[1].bar(models_list, results_summary['Test PRC'], color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('Test AUPRC (rata-rata 12 task)', fontsize=12)
axes[1].set_ylabel('AUPRC Score')
axes[1].set_xlabel('Model')
axes[1].set_ylim(0.0, 0.8)
for bar, val in zip(bars2, results_summary['Test PRC']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### 📈 Cell 17 — Visualisasi 2: Kurva Pelatihan (Learning Curve)

**Apa yang ditampilkan:** Plot garis Valid ROC-AUC vs Epoch untuk GCN (biru) dan GAT (oranye).

#### 📖 Cara baca grafik

**Sumbu X:** epoch (jumlah lewatan training data). Di sini dievaluasi setiap 10 epoch → titik di 10, 20, 30, 40, 50, 60.

**Sumbu Y:** Valid ROC-AUC pada validation set di epoch tersebut.

**Marker bulat biru (●):** snapshot performa GCN.
**Marker kotak oranye (■):** snapshot performa GAT.

**Garis putus-putus vertikal:** menandai **epoch terbaik** untuk masing-masing model (checkpoint yang akhirnya dipakai).

#### 🎯 Apa yang kita cari?

1. **Kurva naik konsisten** → model masih belajar
2. **Kurva mulai datar (plateau)** → mendekati konvergensi
3. **Kurva turun setelah peak** → overfitting (validasi memburuk meski training masih lanjut)
4. **GCN vs GAT**: kurva mana yang lebih tinggi & mencapai peak lebih cepat?

> 💡 **Sinyal early stopping bekerja:** kalau ada penurunan setelah peak tapi kita masih restore epoch terbaik (garis putus-putus), itu artinya kita "menyelamatkan" model dari overfitting.

#### ⚠️ Tanda bahaya
- Kurva naik-turun erratic (noise tinggi) → learning rate terlalu besar
- Kurva flat dari awal → model tidak belajar (mungkin learning rate terlalu kecil atau ada bug)


In [ ]:
# Visualisasi 3: Kurva pelatihan - Valid ROC-AUC per epoch untuk GCN dan GAT
fig, ax = plt.subplots(figsize=(10, 5))

# Mengekstrak data riwayat pelatihan dari masing-masing model
gcn_epochs = [h['epoch'] for h in gcn_history]
gcn_rocs   = [h['valid_roc'] for h in gcn_history]
gat_epochs = [h['epoch'] for h in gat_history]
gat_rocs   = [h['valid_roc'] for h in gat_history]

ax.plot(gcn_epochs, gcn_rocs, 'o-', color='#4C72B0', linewidth=2, markersize=7, label='GCN')
ax.plot(gat_epochs, gat_rocs, 's-', color='#DD8452', linewidth=2, markersize=7, label='GAT')

# Menandai epoch terbaik dengan garis vertikal putus-putus
best_gcn_idx = gcn_rocs.index(max(gcn_rocs))
best_gat_idx = gat_rocs.index(max(gat_rocs))
ax.axvline(x=gcn_epochs[best_gcn_idx], color='#4C72B0', linestyle='--', alpha=0.5,
            label=f'GCN best epoch ({gcn_epochs[best_gcn_idx]})')
ax.axvline(x=gat_epochs[best_gat_idx], color='#DD8452', linestyle='--', alpha=0.5,
            label=f'GAT best epoch ({gat_epochs[best_gat_idx]})')

ax.set_title('Kurva Pelatihan: Valid ROC-AUC vs Epoch', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Valid ROC-AUC (rata-rata 12 task)')
ax.set_xticks(gcn_epochs)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 Cell 18 — Hitung ROC-AUC Per Task

**Apa yang dilakukan:**
1. Ambil prediksi mentah `model.predict(test_dataset)` → array shape `[n_samples, 12, 2]` (prob kelas 0 & 1)
2. Untuk setiap dari 12 task, hitung ROC-AUC **hanya pada sampel dengan label valid** (`w > 0`)
3. Simpan hasil ke DataFrame `per_task_df`

**Kenapa per-task?** Rata-rata umum bisa menyembunyikan fakta bahwa model sangat bagus di task A tapi sangat buruk di task B. Per-task breakdown menunjukkan **kekuatan & kelemahan spesifik**.

**Variabel penting hasil cell:**
- `per_task_df` — DataFrame, baris = task, kolom = model (GCN/GAT)
- `gcn_probs`, `gat_probs` — array probabilitas mentah untuk plot ROC/PRC nanti
- `y_true`, `w_test` — ground truth + bobot untuk masking


In [ ]:
# Menghitung ROC-AUC per task menggunakan prediksi mentah dari model
# model.predict() mengembalikan array [n_samples, n_tasks, 2] (prob kelas 0 dan kelas 1)
print("Menghitung metrik per task...")

# Mengambil label dan bobot dari test dataset
y_true = test_dataset.y   # shape: [n_samples, n_tasks]
w_test = test_dataset.w   # bobot: 0 = data label hilang, >0 = label valid

# Mengambil prediksi probabilitas dari masing-masing model
gcn_preds = gcn_model.predict(test_dataset)  # [n_samples, n_tasks, 2]
gat_preds = gat_model.predict(test_dataset)

# Mengekstrak probabilitas kelas positif (toksik)
gcn_probs = gcn_preds[:, :, 1]  # [n_samples, n_tasks]
gat_probs = gat_preds[:, :, 1]

# Menghitung ROC-AUC per task, hanya untuk sampel dengan label valid (weight > 0)
per_task_roc = {'Task': tasks}
for model_name, probs in [('GCN', gcn_probs), ('GAT', gat_probs)]:
    task_rocs = []
    for t_idx in range(len(tasks)):
        mask = w_test[:, t_idx] > 0
        if mask.sum() > 0 and len(np.unique(y_true[mask, t_idx])) > 1:
            score = roc_auc_score(y_true[mask, t_idx], probs[mask, t_idx])
        else:
            score = np.nan  # Tidak cukup data untuk menghitung ROC-AUC
        task_rocs.append(score)
    per_task_roc[model_name] = task_rocs

# Menyusun tabel per task
per_task_df = pd.DataFrame(per_task_roc).set_index('Task')

print("\nROC-AUC per Task (Test Set):")
print("-" * 35)
print(per_task_df.round(4).to_string())

### 🗺️ Cell 19 — Visualisasi 3: Heatmap ROC-AUC Per Task

**Apa yang ditampilkan:** Matriks berwarna — baris = 12 task, kolom = 2 model (GCN/GAT). Setiap sel diwarnai berdasarkan ROC-AUC dan dianotasi dengan nilai 3 desimal.

#### 📖 Cara baca heatmap

**Skema warna `RdYlGn` (Red-Yellow-Green):**
- 🟢 **Hijau tua (≈ 1.0)** — model sangat baik di task ini
- 🟡 **Kuning (≈ 0.75)** — performa sedang/lumayan
- 🔴 **Merah (≈ 0.5)** — performa setara random/buruk

**Skala warna:** `vmin=0.5, vmax=1.0` — artinya 0.5 (random) → merah pekat, 1.0 (sempurna) → hijau pekat. Nilai di bawah 0.5 (lebih buruk dari random) **tidak digambar** karena di-clamp.

**Angka di tiap sel:** ROC-AUC persis, gampang dibaca tanpa nebak dari warna saja.

#### 🎯 Apa yang kita cari?

1. **Task termudah** — sel paling hijau (biasanya `NR-AhR`, `SR-MMP` di Tox21)
2. **Task tersulit** — sel paling merah (biasanya `SR-HSE`, `NR-PPAR-gamma`)
3. **Konsistensi antar model**: kalau GCN & GAT sama-sama jelek di task tertentu → mungkin task itu memang sulit (data minim, ambigu), bukan kesalahan modelnya
4. **Selisih GCN vs GAT per task**: kadang GAT menang di task tertentu meski rata-rata kalah — bisa jadi kandidat untuk **ensemble**

> 💡 **Insight pendidikan:** Heatmap ini membantu memilih task mana yang prediksinya bisa dipercaya untuk aplikasi nyata. Jangan pakai prediksi dari task dengan AUC < 0.65 untuk pengambilan keputusan.


In [ ]:
# Visualisasi 4: Heatmap ROC-AUC per task untuk setiap model
fig, ax = plt.subplots(figsize=(7, 9))

sns.heatmap(
    per_task_df,
    annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.5, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax
)

ax.set_title('Heatmap ROC-AUC per Task (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('Task Toksikologi', fontsize=11)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

### 🏆 Cell 20 — Pilih Model Terbaik

**Apa yang dilakukan:** Membandingkan `gcn_scores['test']['roc']` vs `gat_scores['test']['roc']`, pilih yang tertinggi sebagai `best_model_name`. Ambil `best_probs` (probabilitas test set) dari model itu untuk dipakai di plot kurva ROC/PRC berikutnya.

Variabel `best_model_name` = "GCN" atau "GAT" (string).


In [ ]:
# Menentukan model GNN terbaik berdasarkan Test ROC-AUC tertinggi
gnn_rows = results_summary[results_summary['Model'].isin(['GCN', 'GAT'])]
best_model_name = gnn_rows.loc[gnn_rows['Test ROC'].idxmax(), 'Model']
best_probs      = gcn_probs if best_model_name == 'GCN' else gat_probs

print(f"Model GNN terbaik (Test ROC-AUC tertinggi): {best_model_name}")
print(f"Test ROC-AUC: {gnn_rows[gnn_rows['Model']==best_model_name]['Test ROC'].values[0]:.4f}")

### 📈 Cell 21 — Visualisasi 4: Kurva ROC Per Task (6 panel)

**Apa yang ditampilkan:** Grid 2×3 = **6 panel kurva ROC** untuk model terbaik:
- **Baris atas:** 3 task **TERBAIK** (ROC-AUC tertinggi) — biasanya yang model paling kuasai
- **Baris bawah:** 3 task **TERBURUK** — yang paling menantang

#### 📖 Cara baca kurva ROC

**Sumbu X — FPR (False Positive Rate):** seberapa sering model salah menandai molekul aman sebagai toksik (skala 0–1; 0 = tidak pernah, 1 = selalu).

**Sumbu Y — TPR (True Positive Rate / Sensitivity / Recall):** seberapa sering model berhasil menangkap molekul toksik yang sebenarnya toksik (skala 0–1).

**Garis biru solid:** kurva ROC model — setiap titik = trade-off pada threshold prediksi berbeda.

**Garis diagonal putus-putus (k--):** referensi *random classifier* (TPR = FPR). Model **harus di atas** garis ini.

**Legenda `AUC=0.xxx`:** Area di bawah kurva = ROC-AUC numerik untuk task ini.

#### 🎯 Apa yang kita cari?

1. **Kurva mendekati pojok kiri atas (0, 1)** = ideal — TPR tinggi tanpa FPR tinggi
2. **Kurva mengikuti diagonal** = model tidak bisa membedakan → tidak berguna
3. **"Knee" curve tajam** = ada threshold optimal yang memisahkan kelas dengan baik
4. **Kurva smooth & gradual** = perbedaan antar threshold halus

#### 💡 Interpretasi pendidikan

- Kurva top-3 yang **AUC > 0.85**: model layak dipakai untuk skrining kandidat
- Kurva bottom-3 yang **AUC < 0.65**: model belum bisa diandalkan untuk task ini — kemungkinan butuh lebih banyak data berlabel atau arsitektur lain

> 🎯 **Trade-off TPR vs FPR**: titik mana di kurva yang kita pilih sebagai threshold tergantung aplikasi. Untuk drug screening kita lebih takut false negative (lolosin obat toksik), jadi pilih threshold yang menjamin TPR tinggi walau FPR ikut naik.


In [ ]:
# Visualisasi 5: Kurva ROC per task - 3 task terbaik + 3 terburuk untuk model GNN terbaik
# Pilih task berdasarkan ROC-AUC pada test set untuk fokus ke diferensiasi paling informatif
scores = per_task_df[best_model_name].dropna()
top3 = scores.nlargest(3).index.tolist()
bot3 = scores.nsmallest(3).index.tolist()
tasks_to_plot = top3 + bot3

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Kurva ROC per Task - Model {best_model_name} (3 task terbaik + 3 terburuk)',
             fontsize=14, fontweight='bold')

for task, ax in zip(tasks_to_plot, axes.flatten()):
    t_idx = tasks.index(task)
    # Hanya menggunakan sampel dengan label valid (bobot > 0)
    mask = w_test[:, t_idx] > 0
    y_t  = y_true[mask, t_idx]
    p_t  = best_probs[mask, t_idx]

    if len(np.unique(y_t)) > 1:
        fpr, tpr, _ = roc_curve(y_t, p_t)
        roc_score   = auc(fpr, tpr)
        ax.plot(fpr, tpr, color='#4C72B0', linewidth=2, label=f'AUC={roc_score:.3f}')
    else:
        ax.text(0.5, 0.5, 'Hanya 1 kelas', ha='center', va='center', transform=ax.transAxes)

    # Garis diagonal sebagai referensi random classifier
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(task, fontsize=10, fontweight='bold')
    ax.set_xlabel('FPR', fontsize=9)
    ax.set_ylabel('TPR', fontsize=9)
    ax.legend(fontsize=9)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()


### 📈 Cell 22 — Visualisasi 5: Kurva Precision-Recall Per Task (6 panel)

**Apa yang ditampilkan:** Sama layout dengan Cell 21 — grid 2×3 = 6 panel — tapi sekarang **Precision-Recall** curve.

#### 📖 Cara baca kurva Precision-Recall

**Sumbu X — Recall:** dari semua molekul yang **benar-benar toksik**, berapa proporsi yang berhasil dideteksi model? (= TPR)

**Sumbu Y — Precision:** dari semua molekul yang **diprediksi toksik**, berapa proporsi yang benar-benar toksik?

**Garis oranye solid:** kurva PR model.

**Garis putus-putus abu-abu horizontal:** baseline = **prevalensi positif** dalam data (mis. kalau 10% molekul toksik, baseline = 0.10). Model harus di **atas** garis ini.

**Legenda `AUPRC=0.xxx`:** area di bawah kurva.

#### 🎯 Kenapa PR Curve penting di samping ROC?

Untuk data **tidak seimbang** (seperti Tox21 dengan ~10% positif), **ROC bisa terlihat bagus tapi PR menyedihkan**. Contoh:
- Model selalu prediksi negatif → ROC = 0.5, tapi tidak diketahui dari kurva ROC seberapa buruk untuk menangkap positif
- PR Curve langsung menunjukkan: kalau kurva sangat dekat baseline → model gagal di kelas minoritas

#### 🎯 Apa yang kita cari?

1. **Kurva tinggi di kiri (Recall=0)** = saat threshold ketat, model precise
2. **Kurva turun saat Recall→1** = trade-off normal (semakin banyak yang ditangkap, semakin banyak false positive)
3. **Kurva jauh di atas baseline** = model lebih baik dari random
4. **Kurva nempel baseline** = model **tidak berguna untuk kelas minoritas** meski ROC-AUC mungkin terlihat OK

> 💡 **Pelajaran utama:** Jangan hanya percaya ROC-AUC. Untuk dataset imbalanced seperti Tox21, AUPRC + baseline-nya jauh lebih informatif tentang apakah model benar-benar bisa menangkap molekul toksik (yang kita peduli).


In [ ]:
# Visualisasi 6: Kurva Precision-Recall per task - 3 task terbaik + 3 terburuk
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Kurva Precision-Recall per Task - Model {best_model_name} (3 task terbaik + 3 terburuk)',
             fontsize=14, fontweight='bold')

for task, ax in zip(tasks_to_plot, axes.flatten()):
    t_idx = tasks.index(task)
    mask = w_test[:, t_idx] > 0
    y_t  = y_true[mask, t_idx]
    p_t  = best_probs[mask, t_idx]

    # Prevalensi positif = baseline AUPRC untuk random classifier
    prevalence = y_t.mean() if len(y_t) > 0 else 0

    if len(np.unique(y_t)) > 1:
        prec, rec, _ = precision_recall_curve(y_t, p_t)
        prc_score    = auc(rec, prec)
        ax.plot(rec, prec, color='#DD8452', linewidth=2, label=f'AUPRC={prc_score:.3f}')
        ax.axhline(y=prevalence, color='gray', linestyle='--', linewidth=0.8,
                   label=f'Baseline={prevalence:.3f}')
    else:
        ax.text(0.5, 0.5, 'Hanya 1 kelas', ha='center', va='center', transform=ax.transAxes)

    ax.set_title(task, fontsize=10, fontweight='bold')
    ax.set_xlabel('Recall', fontsize=9)
    ax.set_ylabel('Precision', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()


## Analisis Per Task

### 📋 Cell 24 — Analisis Per-Task Top/Bottom

**Apa yang dilakukan:** Mengurutkan `per_task_df` berdasarkan ROC-AUC model terbaik dan mencetak ranking lengkap.

**Output:** daftar task dari yang paling mudah ke paling sulit untuk model. Berguna untuk diskusi: kenapa task A susah? Apakah data label terlalu sedikit? Apakah biologi-nya kompleks?


In [ ]:
# Mengidentifikasi task dengan performa terbaik dan terburuk untuk model terbaik
# Mengurutkan tabel dari ROC-AUC tertinggi ke terendah
pertask_sorted = per_task_df[[best_model_name]].copy()
pertask_sorted.columns = ['ROC-AUC']
pertask_sorted = pertask_sorted.sort_values('ROC-AUC', ascending=False)

print(f"Performa per Task - Model {best_model_name} (Test Set, diurutkan dari terbaik):")
print("-" * 40)
for task, row in pertask_sorted.iterrows():
    print(f"  {task:20s}: {row['ROC-AUC']:.4f}")

# Mengidentifikasi 3 task terbaik dan 3 terburuk
valid_scores = per_task_df[best_model_name].dropna()
best_tasks   = valid_scores.nlargest(3)
worst_tasks  = valid_scores.nsmallest(3)

print(f"\n3 Task TERBAIK ({best_model_name}):")
for task, score in best_tasks.items():
    print(f"  {task:20s}: {score:.4f}")

print(f"\n3 Task TERBURUK ({best_model_name}):")
for task, score in worst_tasks.items():
    print(f"  {task:20s}: {score:.4f}")

## Export Model Artifacts untuk API Backend

Menyimpan daftar task dan metadata model agar FastAPI backend bisa load model dan menampilkan info ke frontend tanpa hardcode.

### 💾 Cell 26 — Export Artifacts untuk Backend FastAPI

**Apa yang dilakukan:** Menulis beberapa file ke folder `checkpoints/`:
1. **`tasks.json`** — daftar 12 task + deskripsi singkat
2. **`model_info.json`** — hyperparameter, metrik, dan path checkpoint untuk GCN & GAT, plus `per_task_test_scores`
3. **`train_fingerprints.npz`** — Morgan fingerprint (radius=2, 2048-bit) untuk semua molekul training set
4. **`checkpoints/gcn_best/` dan `gat_best/`** — bobot model PyTorch (sudah disimpan di cell training)

**Kenapa dibutuhkan:** Backend FastAPI (`backend/main.py`) membaca file-file ini saat startup untuk:
- Load model GCN + GAT
- Tampilkan reliability per task di frontend (dari `per_task_test_scores`)
- **Applicability Domain check** — bandingkan molekul query dengan training set via Tanimoto similarity (`train_fingerprints.npz`)

> 💡 **Tanpa cell ini**, backend masih jalan tapi fitur Applicability Domain dan reliability stars di frontend akan disabled.


In [ ]:
# Menulis daftar 12 task dan metadata model ke folder checkpoints/
# File ini dibaca backend FastAPI saat startup untuk inisialisasi.
import json as _json

project_root = os.path.dirname(os.path.abspath(os.getcwd() + '/Tox21.ipynb'))
checkpoints_dir = os.path.join(project_root, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)

# Daftar 12 task dengan deskripsi singkat untuk konteks frontend
task_descriptions = {
    'NR-AR':         'Androgen Receptor - disrupsi hormon reproduktif',
    'NR-AR-LBD':     'Androgen Receptor LBD - domain pengikatan ligand',
    'NR-AhR':        'Aryl hydrocarbon Receptor - aktivasi prokarsinogen',
    'NR-Aromatase':  'CYP19A1 - konversi androgen ke estrogen',
    'NR-ER':         'Estrogen Receptor - endocrine disruptor',
    'NR-ER-LBD':     'Estrogen Receptor LBD - domain pengikatan ligand',
    'NR-PPAR-gamma': 'PPAR-gamma - regulasi diferensiasi adiposit',
    'SR-ARE':        'Antioxidant Response Element - stres oksidatif',
    'SR-ATAD5':      'ATAD5 - kerusakan DNA',
    'SR-HSE':        'Heat Shock - misfolding protein',
    'SR-MMP':        'Mitochondrial Membrane Potential - prekursor apoptosis',
    'SR-p53':        'p53 - kerusakan DNA berat',
}

with open(os.path.join(checkpoints_dir, 'tasks.json'), 'w') as f:
    _json.dump({'tasks': list(tasks), 'descriptions': task_descriptions}, f, indent=2)

# Metadata model GCN (best model)
gcn_best_epoch = max(gcn_history, key=lambda h: h['valid_roc'])['epoch']
gat_best_epoch = max(gat_history, key=lambda h: h['valid_roc'])['epoch']

model_info = {
    'best_model': best_model_name,
    'gcn': {
        'checkpoint_dir': 'checkpoints/gcn_best',
        'best_epoch': gcn_best_epoch,
        'train_roc_auc': float(gcn_scores['train']['roc']),
        'valid_roc_auc': float(gcn_scores['valid']['roc']),
        'test_roc_auc':  float(gcn_scores['test']['roc']),
        'test_prc_auc':  float(gcn_scores['test']['prc']),
        'hyperparameters': {
            'graph_conv_layers': [128, 128],
            'dropout': 0.3,
            'learning_rate': 0.001,
            'batch_size': 64,
        },
    },
    'gat': {
        'checkpoint_dir': 'checkpoints/gat_best',
        'best_epoch': gat_best_epoch,
        'train_roc_auc': float(gat_scores['train']['roc']),
        'valid_roc_auc': float(gat_scores['valid']['roc']),
        'test_roc_auc':  float(gat_scores['test']['roc']),
        'test_prc_auc':  float(gat_scores['test']['prc']),
        'hyperparameters': {
            'n_attention_heads': 4,
            'dropout': 0.3,
            'learning_rate': 0.001,
            'batch_size': 64,
        },
    },
    'featurizer': 'MolGraphConvFeaturizer(use_edges=True)',
    'n_tasks': len(tasks),
    'training': {
        'epochs_total': 60,
        'check_every': 10,
        'balancing_transformer': True,
        'split': 'random 80/10/10',
    },
}

with open(os.path.join(checkpoints_dir, 'model_info.json'), 'w') as f:
    _json.dump(model_info, f, indent=2)

print(f'[OK] Artifacts tersimpan di: {checkpoints_dir}')
print(f'      - tasks.json       ({len(tasks)} tasks)')
print(f'      - model_info.json  (best model: {best_model_name})')
print(f'      - {model_info["gcn"]["checkpoint_dir"]}/  (weights GCN)')
print(f'      - {model_info["gat"]["checkpoint_dir"]}/  (weights GAT)')

# ─── Per-task test scores (untuk display reliabilitas di frontend) ────────────
gcn_per_task_scores = {
    task: float(per_task_df.loc[task, 'GCN'])
    for task in tasks
    if not np.isnan(per_task_df.loc[task, 'GCN'])
}
gat_per_task_scores = {
    task: float(per_task_df.loc[task, 'GAT'])
    for task in tasks
    if not np.isnan(per_task_df.loc[task, 'GAT'])
}
model_info['gcn']['per_task_test_scores'] = gcn_per_task_scores
model_info['gat']['per_task_test_scores'] = gat_per_task_scores

# Simpan ulang model_info.json dengan per-task scores
with open(os.path.join(checkpoints_dir, 'model_info.json'), 'w') as f:
    _json.dump(model_info, f, indent=2)

# ─── Export training set fingerprints untuk Applicability Domain check ────────
from rdkit.Chem import AllChem
from rdkit import Chem as _Chem
from rdkit.DataStructs import ConvertToNumpyArray as _ConvertToNumpyArray

print('[EXPORT] Menghitung Morgan fingerprint training set...')
train_fps_list = []
train_smiles_list = []
for smi in train_dataset.ids:
    mol = _Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    arr = np.zeros(2048, dtype=np.uint8)
    _ConvertToNumpyArray(fp, arr)
    train_fps_list.append(arr)
    train_smiles_list.append(smi)

train_fps_array = np.array(train_fps_list, dtype=np.uint8)
np.savez_compressed(
    os.path.join(checkpoints_dir, 'train_fingerprints.npz'),
    fps=train_fps_array,
    smiles=np.array(train_smiles_list)
)
print(f'[OK] train_fingerprints.npz tersimpan: {train_fps_array.shape[0]} molekul × {train_fps_array.shape[1]} bit')
print(f'      - per_task_test_scores ditambahkan ke model_info.json')


## Diskusi: GNN vs MLP untuk Data Molekuler

### Mengapa GNN Lebih Efektif dari MLP untuk Prediksi Toksisitas?

Mayr et al. (2016) dalam **DeepTox: Toxicity Prediction using Deep Learning** menunjukkan bahwa
arsitektur deep learning yang dapat menangkap representasi hierarkis dari data kimia secara
konsisten mengungguli metode berbasis fingerprint tradisional.

**Keterbatasan MLP dengan fingerprint (CircularFingerprint/ECFP):**
- Representasi ECFP adalah vektor biner tetap yang kehilangan informasi spasial tentang konektivitas atom
- Dua molekul berbeda dapat memiliki fingerprint identik (*hash collision*)
- MLP tidak bisa memanfaatkan struktur graf yang inheren dalam molekul
- Overfitting parah terlihat dari gap Train ROC (0.99) vs Test ROC (0.75) pada baseline MLP

**Keunggulan GNN (GCN/GAT):**
- Bekerja langsung pada representasi graf molekul — atom sebagai *node*, ikatan sebagai *edge*
- GCN mengagregasi informasi dari atom tetangga secara iteratif, menangkap konteks kimia lokal
- GAT menggunakan *attention mechanism* untuk memberi bobot berbeda pada tetangga yang lebih
  relevan secara kimiawi (gugus fungsi reaktif mendapat bobot lebih tinggi)
- Pembelajaran representasi bersifat *end-to-end* tanpa rekayasa fitur manual
- Train-test gap yang lebih kecil = generalisasi lebih baik

### Ketidakseimbangan Kelas dan Pentingnya AUPRC

Cavasotto & Scardino (2022) dalam **In Silico Drug Discovery and Design** menekankan bahwa
dataset toksikologi seperti Tox21 memiliki karakteristik **ketidakseimbangan kelas yang parah**:
proporsi molekul toksik (positif) jauh lebih sedikit dari non-toksik (sekitar 3–15%).

**Mengapa ROC-AUC bisa menyesatkan pada data imbalanced:**
- ROC-AUC dihitung dari TPR (True Positive Rate) vs FPR (False Positive Rate)
- Dengan kelas negatif yang sangat banyak, FPR tetap rendah meskipun banyak prediksi False Positive
- Model yang bias ke kelas negatif dapat memperoleh ROC-AUC tinggi secara artifisial

**Mengapa AUPRC lebih informatif:**
- Precision-Recall curve tidak melibatkan True Negative sama sekali
- AUPRC sensitif terhadap kemampuan model mendeteksi kelas positif yang langka
- Baseline AUPRC untuk random classifier = *prevalensi* kelas positif (misal 10%), bukan 0.5
- AUPRC yang rendah mengindikasikan model kesulitan mendeteksi molekul toksik secara tepat

**Strategi yang diterapkan dalam notebook ini:**
- `BalancingTransformer` menyesuaikan bobot sampel untuk mengurangi bias ke kelas negatif
- Pelaporan AUPRC sejajar ROC-AUC memberikan gambaran performa yang lebih lengkap dan jujur

### Keterbatasan dan Peningkatan yang Mungkin Dilakukan

**Keterbatasan saat ini:**
1. **GTX 1650 VRAM terbatas (4 GB)** — batch size kecil (64) dan model dengan hidden dim besar
   akan terkena OOM. Speedup GPU tidak signifikan untuk dataset kecil (6K molekul, molekul kecil)
   karena overhead transfer CPU↔GPU melebihi manfaat paralelisme GPU
2. **Hanya 60 epoch** — beberapa GNN memerlukan >200 epoch untuk konvergensi penuh;
   dengan GPU tersedia, menambah epoch tidak menambah waktu secara signifikan
3. **Hyperparameter default** — belum dioptimasi melalui grid search atau Bayesian optimization
4. **DGL 2.4.0 (bukan 2.5.0)** — versi terbaru di CDN DGL untuk folder torch-2.5 masih HTTP 403;
   versi 2.4.0 sudah stabil dan kompatibel dengan DeepChem 2.8

**Peningkatan yang direkomendasikan:**
1. **Tambah epoch** — coba 150–200 epoch, early stopping akan mencegah overfitting
2. **Naikkan batch_size** — coba 128 atau 256 untuk utilisasi GPU lebih optimal
3. **AttentiveFPModel** (Xiong et al. 2020) — model GNN state-of-the-art untuk prediksi
   properti molekuler; kini bisa dijalankan dengan DGL CUDA yang sudah terpasang
4. **Ensemble model** — rata-rata prediksi GCN + GAT biasanya meningkatkan ~1-2% ROC-AUC
5. **Optuna hyperparameter tuning** — cari learning rate, hidden dim, n_layers optimal
6. **Transfer learning** — pre-training pada dataset lebih besar (ChEMBL) lalu fine-tuning pada Tox21
7. **Per-task threshold tuning** — optimalkan threshold klasifikasi per task sesuai konteks klinis

### 🧪 Cell 30 — Demo Prediksi pada SMILES Baru

**Apa yang dilakukan:** Memprediksi toksisitas untuk 5 senyawa contoh (Aspirin, Caffeine, Paracetamol, Benzene, Ethanol) menggunakan **model terbaik**.

**Alur prediksi:**
1. Featurize SMILES → graph (pakai `MolGraphConvFeaturizer` yang sama dengan training)
2. Bungkus ke `NumpyDataset` (DeepChem mengharuskan format ini)
3. `model.predict()` → array probabilitas per task
4. Print top-3 endpoint dengan probabilitas tertinggi per molekul

**Hasil yang diharapkan:**
- **Aspirin, Caffeine, Paracetamol** → probabilitas rendah di semua endpoint (memang aman pada dosis wajar)
- **Benzene** → probabilitas tinggi di `NR-AhR` (Benzene memang aktivator AhR yang kuat — karsinogen kelas 1 IARC)
- **Ethanol** → probabilitas moderat di beberapa endpoint

> 🎯 **Cell ini = "smoke test"** bahwa model terlatih masuk akal secara biologis sebelum dipakai di aplikasi nyata.


In [ ]:
# Memprediksi toksisitas untuk 5 contoh molekul menggunakan model terbaik
# Demonstrasi penggunaan model untuk prediksi senyawa baru
print(f"Prediksi toksisitas menggunakan model: {best_model_name}")
print("=" * 60)

# Daftar SMILES contoh dan nama senyawanya
example_smiles = {
    'Aspirin':     'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine':    'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Benzene':     'c1ccccc1',
    'Ethanol':     'CCO',
}

# Menggunakan featurizer yang sama dengan training
featurizer  = dc.feat.MolGraphConvFeaturizer(use_edges=True)
smiles_list = list(example_smiles.values())
features    = featurizer.featurize(smiles_list)

# Membuat dataset dummy untuk prediksi (label tidak diperlukan)
pred_dataset = dc.data.NumpyDataset(
    X=features,
    y=np.zeros((len(smiles_list), len(tasks))),
    ids=smiles_list
)

# Memilih model terbaik untuk prediksi
best_model = gcn_model if best_model_name == 'GCN' else gat_model

# Melakukan prediksi - output adalah probabilitas kelas toksik
predictions  = best_model.predict(pred_dataset)  # [n_mol, n_tasks, 2]
proba_toxic  = predictions[:, :, 1]              # probabil itas kelas 1 (toksik)

# Menyusun hasil prediksi dalam DataFrame yang mudah dibaca
pred_df = pd.DataFrame(
    proba_toxic,
    index=list(example_smiles.keys()),
    columns=tasks
)

print("\nProbabilitas Toksisitas per Endpoint (0 = aman, 1 = toksik):")
print("-" * 60)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(pred_df.T.to_string())

print("\n" + "="*60)
print("Ringkasan: endpoint dengan probabilitas toksisitas tertinggi per molekul")
print("="*60)
for mol_name, row in pred_df.iterrows():
    most_toxic_task  = row.idxmax()
    most_toxic_score = row.max()
    flag = '(!) PERLU PERHATIAN' if most_toxic_score > 0.5 else ''
    print(f"  {mol_name:12s}: {most_toxic_task:15s} = {most_toxic_score:.3f} {flag}")